[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sk-2103/AI4Sustainability-Labs/blob/main/labs/lab02_random_forest_crop_yield/lab02_evaluation.ipynb)  
**AI for Sustainability** · University of Wisconsin–Madison · Labs by Saurabh Kaushik · Course instructor: Dr. Beth Tellman  
*Make a copy (File → Save a copy in Drive) before you start. Part of the [AI for Sustainability lab series](https://github.com/Sk-2103/AI4Sustainability-Labs).*

# Lab 2 — Evaluation Notebook (20 points)
**Topic:** Random Forest for Crop Yield Prediction (FAOSTAT + World Bank)  

---

## IMPORTANT: Make your own copy first (so everyone has a unique notebook)

You are working in **Google Colab**, which is shared.

### Step 1 — Download a copy to your computer
1. In Colab, go to **File → Download → Download .ipynb**
2. Save it with this filename format:

**`Lab2_Eval_<YourNetID>_<LastName>.ipynb`**

Example: `Lab2_Eval_sk1234_Kaushik.ipynb`

### Step 2 — Re-upload your renamed copy
1. In Colab, go to **File → Upload notebook**
2. Upload your renamed file
3. Do **all edits in your renamed copy**

### Step 3 — Submit
Submit your final `.ipynb` on the course LMS.

> Do not submit links to a shared notebook. Submit the `.ipynb` file.

---

## Grading (Total = 20 points)

### Part A — Theory (10 points)
5 questions × 2 points each.

### Part B — Practical (10 points)
Re-create the full data pipeline and then train **three Random Forest variants** by changing hyperparameters in three blank cells.

---



## Setup


In [ ]:

# If needed (uncomment once in Colab):
# !pip -q install pandas numpy scikit-learn matplotlib wbgapi requests

import os
import io
import math
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
!pip install wbgapi
import wbgapi as wb

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor


## Configuration

This evaluation reproduces the same pipeline as the lab:

- FAOSTAT bulk download (QCL)  
- Compute maize yield as: **yield (t/ha) = production (t) / area harvested (ha)**  
- Fetch World Bank predictors using `wbgapi`  
- Merge into a modeling table


In [ ]:

# -----------------------
# CONFIG (do not change unless instructed)
# -----------------------
CROP_NAME = "Maize (corn)"
FAOSTAT_ITEM_CODE = 56

# Element codes discovered from your FAOSTAT bulk CSV (QCL normalized):
PROD_CODE = 5510   # Production (t)
AREA_CODE = 5312   # Area harvested (ha)

START_YEAR = 2000
END_YEAR   = 2022

COUNTRIES_ISO3 = ["USA", "IND", "BRA", "NGA", "FRA", "ETH", "MEX", "ZAF"]

# World Bank indicator codes
WB_INDICATORS = {
    "precip_mm":    "AG.LND.PRCP.MM",
    "gdp_pc":       "NY.GDP.PCAP.CD",
    "pop":          "SP.POP.TOTL.CD" if False else "SP.POP.TOTL",  # keep SP.POP.TOTL; line kept to show students codes are strings
    "ag_value_add": "NV.AGR.TOTL.CD",
}

OUT_CSV = "faostat_worldbank_maize_yield_small.csv"
CACHE_DIR = "cache_faostat"
os.makedirs(CACHE_DIR, exist_ok=True)


## Step 1 — Download FAOSTAT bulk data (cached) and compute yield

Teaching note: the ZIP/CSV is large. This notebook **caches** the downloaded ZIP in `cache_faostat/`.
If you re-run, it should use the cached file.


In [ ]:

BULK_BASE = "https://fenixservices.fao.org/faostat/static/bulkdownloads"
CANDIDATE_ZIPS = [
    "Production_Crops_Livestock_E_All_Data_(Normalized).zip",
    "Production_Crops_Livestock_E_All_Data.zip",
    "Production_Crops_Livestock_E_All_Data_NOFLAG.zip",
    "Production_Crops_E_All_Data_(Normalized).zip",
    "Production_Crops_E_All_Data.zip",
]

def download_first_working_zip(base_url: str, names: list, cache_dir: str, timeout=300):
    last_err = None
    for name in names:
        url = f"{base_url}/{name}"
        local_path = Path(cache_dir) / name

        if local_path.exists() and local_path.stat().st_size > 0:
            print(f"[CACHE] Using cached FAOSTAT ZIP: {local_path}")
            return url, local_path

        try:
            head = requests.head(url, timeout=30, allow_redirects=True)
            if head.status_code != 200:
                print(f"[SKIP] {name}: HEAD status {head.status_code}")
                continue

            print(f"[OK] Downloading: {url}")
            data = requests.get(url, timeout=timeout).content
            local_path.write_bytes(data)
            print(f"[SAVED] {local_path} ({local_path.stat().st_size/1e6:.1f} MB)")
            return url, local_path

        except Exception as e:
            last_err = e
            print(f"[SKIP] {name}: {type(e).__name__}: {e}")

    raise RuntimeError(f"No working FAOSTAT bulk ZIP found. Last error: {last_err}")

bulk_url, zip_path = download_first_working_zip(BULK_BASE, CANDIDATE_ZIPS, CACHE_DIR)
print("Using bulk file:", bulk_url)


In [ ]:

def read_largest_csv_from_zip(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise RuntimeError("No CSV found inside the FAOSTAT bulk ZIP.")
        csv_name = max(csv_names, key=lambda n: zf.getinfo(n).file_size)
        print("Reading CSV:", csv_name, f"({zf.getinfo(csv_name).file_size/1e6:.1f} MB)")
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, low_memory=False)
    return df

fao = read_largest_csv_from_zip(zip_path)
print("Loaded rows:", len(fao))
print("Columns:", fao.columns.tolist())


In [ ]:

# FAOSTAT bulk CSV does not include ISO3, so we map ISO3 -> FAOSTAT 'Area' name.
ISO3_TO_FAO_AREA = {
    "USA": "United States of America",
    "IND": "India",
    "BRA": "Brazil",
    "NGA": "Nigeria",
    "FRA": "France",
    "ETH": "Ethiopia",
    "MEX": "Mexico",
    "ZAF": "South Africa",
}
area_to_iso3 = {v: k for k, v in ISO3_TO_FAO_AREA.items()}
wanted_areas = list(ISO3_TO_FAO_AREA.values())

def compute_yield_from_prod_area(fao, item_code, prod_code, area_code, start_year, end_year, wanted_areas):
    required = {"Area","Item Code","Element Code","Year","Value"}
    missing = required - set(fao.columns)
    assert not missing, f"Missing columns in FAOSTAT bulk CSV: {missing}"

    prod = fao[
        (fao["Item Code"] == item_code) &
        (fao["Element Code"] == prod_code) &
        (fao["Area"].isin(wanted_areas)) &
        (fao["Year"].between(start_year, end_year))
    ][["Area", "Year", "Value"]].copy()

    area = fao[
        (fao["Item Code"] == item_code) &
        (fao["Element Code"] == area_code) &
        (fao["Area"].isin(wanted_areas)) &
        (fao["Year"].between(start_year, end_year))
    ][["Area", "Year", "Value"]].copy()

    prod = prod.rename(columns={"Value": "production_t"})
    area = area.rename(columns={"Value": "area_ha"})

    prod["production_t"] = pd.to_numeric(prod["production_t"], errors="coerce")
    area["area_ha"] = pd.to_numeric(area["area_ha"], errors="coerce")

    tmp = prod.merge(area, on=["Area", "Year"], how="inner").dropna(subset=["production_t","area_ha"])
    tmp = tmp[tmp["area_ha"] > 0]

    tmp["yield_t_per_ha"] = tmp["production_t"] / tmp["area_ha"]
    tmp["iso3"] = tmp["Area"].map(area_to_iso3)
    tmp["year"] = tmp["Year"].astype(int)

    out = tmp[["iso3", "year", "yield_t_per_ha"]].dropna()
    return out.sort_values(["iso3","year"]).reset_index(drop=True)

yield_df = compute_yield_from_prod_area(
    fao=fao,
    item_code=FAOSTAT_ITEM_CODE,
    prod_code=PROD_CODE,
    area_code=AREA_CODE,
    start_year=START_YEAR,
    end_year=END_YEAR,
    wanted_areas=wanted_areas
)
print("yield_df rows:", len(yield_df))
yield_df.head()


## Step 2 — Fetch World Bank indicators using `wbgapi` and reshape to long format

Your `wbgapi` version returns data as:
- index: (`economy`, `series`)
- columns: `YR2000`, `YR2001`, ...

We reshape it into:
`iso3 | year | precip_mm | gdp_pc | pop | ag_value_add`


In [ ]:

time_range = [f"YR{y}" for y in range(START_YEAR, END_YEAR + 1)]

wb_raw = wb.data.DataFrame(
    list(WB_INDICATORS.values()),
    economy=COUNTRIES_ISO3,
    time=time_range,
    labels=False,
)

wb_wide = wb_raw.reset_index()  # columns: economy, series, YR2000..YR2022
year_cols = [c for c in wb_wide.columns if str(c).startswith("YR")]

wb_long = wb_wide.melt(
    id_vars=["economy", "series"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

wb_long["year"] = wb_long["year"].str.replace("YR","", regex=False).astype(int)
wb_long = wb_long.rename(columns={"economy":"iso3"})

code_to_short = {v:k for k,v in WB_INDICATORS.items()}
wb_long["feature"] = wb_long["series"].map(code_to_short)
wb_long = wb_long.dropna(subset=["feature"])

wb_df = wb_long.pivot_table(
    index=["iso3","year"],
    columns="feature",
    values="value",
    aggfunc="first"
).reset_index()

for c in code_to_short.values():
    if c in wb_df.columns:
        wb_df[c] = pd.to_numeric(wb_df[c], errors="coerce")

print("wb_df rows:", len(wb_df))
wb_df.head()


## Step 3 — Merge and save the modeling table

This produces the final table with columns:
- `iso3`, `year`, `yield_t_per_ha`
- predictors: `precip_mm`, `gdp_pc`, `pop`, `ag_value_add`


In [ ]:

df = pd.merge(yield_df, wb_df, on=["iso3","year"], how="left").sort_values(["iso3","year"]).reset_index(drop=True)

print("Rows:", len(df))
print("Missingness (top):")
print(df.isna().mean().sort_values(ascending=False).head(10))

df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

df.head()


## Step 4 — Train/test split (same rule as lab)

- Train: years ≤ 2016  
- Test: years > 2016  

We do **not** use `iso3` or `year` as model features.


In [ ]:

FEATURES = ["precip_mm", "gdp_pc", "pop", "ag_value_add"]
TARGET = "yield_t_per_ha"
TRAIN_END_YEAR = 2016

train = df[df["year"] <= TRAIN_END_YEAR].dropna(subset=[TARGET]).copy()
test  = df[df["year"] >  TRAIN_END_YEAR].dropna(subset=[TARGET]).copy()

# Fill missing predictors with TRAIN medians
train_medians = train[FEATURES].median(numeric_only=True)

X_train = train[FEATURES].fillna(train_medians)
y_train = train[TARGET].values

X_test = test[FEATURES].fillna(train_medians)
y_test = test[TARGET].values

print("Train rows:", len(train), "| Test rows:", len(test))


---

# Part A — Theory (10 points)

Answer **in your own words**. Keep answers short but precise (2–6 sentences each).

**Scoring guidance (2 points each):**
- 2: correct + clear
- 1: partially correct
- 0: incorrect/blank


### Q1. Why Random Forest is called *random*?

**Write your answer below.**


In [ ]:
# TODO: Write your answer as a multi-line string.
answer_q1 = """

"""
print(answer_q1)


### Q2. What is the difference between a Random Forest and a Decision Tree?

**Write your answer below.**


In [ ]:
# TODO: Write your answer as a multi-line string.
answer_q2 = """

"""
print(answer_q2)


### Q3. Write any suitable application you can think of in sustainability (1–2 examples).

**Write your answer below.**


In [ ]:
# TODO: Write your answer as a multi-line string.
answer_q3 = """

"""
print(answer_q3)


### Q4. Random Forests can model non-linear relationships. Explain in plain language how a Random Forest makes predictions.

**Write your answer below.**


In [ ]:
# TODO: Write your answer as a multi-line string.
answer_q4 = """

"""
print(answer_q4)


### Q5. What is bootstrapping?

**Write your answer below.**


In [ ]:
# TODO: Write your answer as a multi-line string.
answer_q5 = """

"""
print(answer_q5)


---

# Part B — Practical (10 points)

## Task
Train **three Random Forest variants** by changing hyperparameters in the three cells below.

### Requirements
- Keep using `RandomForestRegressor`
- Make each variant meaningfully different (e.g., `n_estimators`, `max_depth`, `max_features`, `min_samples_leaf`, `bootstrap`)
- Print metrics for each model and fill the results table

### What to submit
- A results table with Baseline + Variant 1/2/3
- A short paragraph summarizing observations


## Helper: evaluation function (do not edit)


In [ ]:

def eval_regression(y_true, y_pred):
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2


## Baseline model (provided)


In [ ]:

baseline = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)
baseline.fit(X_train, y_train)
pred0 = baseline.predict(X_test)
rmse0, mae0, r20 = eval_regression(y_test, pred0)
print("Baseline RF | RMSE:", round(rmse0,3), "MAE:", round(mae0,3), "R2:", round(r20,3))


## Variant 1 (student edits this cell)


In [ ]:

# TODO: Edit hyperparameters to create a meaningful variant
rf_1 = RandomForestRegressor(
    n_estimators= ## choose number of tree
    random_state=42, ## we can keep it same
    min_samples_leaf= ## choose number of leaf
    max_depth= # choose depth  (maximum number of splits (levels) a tree is allowed to grow.)
    n_jobs=-1
)
rf_1.fit(X_train, y_train)
pred1 = rf_1.predict(X_test)
rmse1, mae1, r21 = eval_regression(y_test, pred1)
print("Variant 1 | RMSE:", round(rmse1,3), "MAE:", round(mae1,3), "R2:", round(r21,3))


## Variant 2 (student edits this cell)


In [ ]:
# TODO: Edit hyperparameters to create a meaningful variant
rf_2 = RandomForestRegressor(
    n_estimators=## choose number of tree
    random_state=42, ## we can keep it same
    min_samples_leaf=## choose number of leaf
    max_depth=# choose depth  (maximum number of splits (levels) a tree is allowed to grow.)
    n_jobs=-1
)
rf_2.fit(X_train, y_train)
pred2 = rf_2.predict(X_test)
rmse2, mae2, r22 = eval_regression(y_test, pred2)
print("Variant 2 | RMSE:", round(rmse2,3), "MAE:", round(mae2,3), "R2:", round(r22,3))


## Variant 3 (student edits this cell)


In [ ]:
# TODO: Edit hyperparameters to create a meaningful variant
rf_3 = RandomForestRegressor(
    n_estimators=## choose number of tree
    random_state=42, ## we can keep it same
    min_samples_leaf=## choose number of leaf
    max_depth=# choose depth  (maximum number of splits (levels) a tree is allowed to grow.)
    n_jobs=-1
)
rf_3.fit(X_train, y_train)
pred3 = rf_3.predict(X_test)
rmse3, mae3, r23 = eval_regression(y_test, pred3)
print("Variant 3 | RMSE:", round(rmse3,3), "MAE:", round(mae3,3), "R2:", round(r23,3))


## Results table (do not edit — should run after all variants)


In [ ]:

results = [
    ("Baseline", rmse0, mae0, r20),
    ("Variant 1", rmse1, mae1, r21),
    ("Variant 2", rmse2, mae2, r22),
    ("Variant 3", rmse3, mae3, r23),
]
results_df = pd.DataFrame(results, columns=["model","rmse_t_per_ha","mae_t_per_ha","r2"]).copy()
results_df


## One plot (choose one model)


In [ ]:

# Choose one: pred0, pred1, pred2, pred3
pred_to_plot = pred0

plt.figure()
plt.scatter(y_test, pred_to_plot)
plt.xlabel("True yield (t/ha)")
plt.ylabel("Predicted yield (t/ha)")
plt.title("Predictions vs True (selected model)")
plt.show()


## Observations (2 pts)


In [ ]:

# TODO: Write a short paragraph comparing baseline vs variants using your numbers.
observations = """

"""
print(observations)


---

# Submission checklist
- [ ] Your notebook filename includes your NetID and LastName
- [ ] Theory answers Q1–Q5 are completed
- [ ] Pipeline runs and produces a non-empty merged table
- [ ] Baseline + 3 variants metrics shown in results table
- [ ] One plot generated
- [ ] Observations paragraph written

**Export/Download before submitting:** File → Download → Download .ipynb
